# 03 - Preprocessing Policy

## Objective

The purpose of this notebook is to define and implement the preprocessing strategy for the Adult Income Dataset before model training.

Based on the findings from Data Validation and Exploratory Data Analysis (EDA), this notebook establishes the preprocessing decisions required to prepare the dataset for CatBoost classification.

## Tasks Performed

### 1. Missing Value Handling

* Identified categorical features containing missing values represented by the '?' symbol.
* Replaced missing values with a new category named **'Unknown'** to preserve information and avoid unnecessary data loss.

### 2. Duplicate Record Handling

* Identified duplicate observations in both training and testing datasets.
* Removed duplicate records to improve data quality and prevent repeated observations from influencing model training.

### 3. Outlier Handling Policy

* Reviewed numerical features containing statistical outliers.
* Retained all outliers because they represent valid observations and CatBoost is robust to extreme values.

### 4. Target Variable Encoding

* Converted the income target variable into a binary format:

  * 0 → Income ≤ $50K
  * 1 → Income > $50K

### 5. Categorical Feature Handling

* Evaluated categorical variables for encoding requirements.
* Retained categorical features in their original form to leverage CatBoost's native categorical feature handling capability.

### 6. Feature Scaling Policy

* Assessed the need for feature scaling.
* Determined that scaling is not required because CatBoost is a tree-based algorithm and is insensitive to feature magnitude differences.

### 7. Feature Selection Policy

* Reviewed all predictor variables for potential removal.
* Retained all available features for model training to allow CatBoost to determine feature importance.

## Outcome

At the completion of this notebook, the dataset is cleaned, properly encoded, and prepared for CatBoost model training. The preprocessing decisions established here will be used consistently throughout model training, hyperparameter tuning, evaluation, and responsible AI assessment.


## Missing Value Handling Policy

### Problem Identification

The dataset contains missing values represented by the '?' symbol in the following categorical features:

- workclass
- occupation
- native_country

During data validation, these features were found to contain a small proportion of missing observations.

### Decision

The missing values will be replaced with a new category named **'Unknown'** rather than removing records or imputing with the most frequent category.

### Justification

- Removing affected records would result in unnecessary data loss.
- Replacing missing values with the mode could distort the original category distribution.
- Creating an 'Unknown' category preserves information about missingness and allows the model to learn potential patterns associated with missing values.
- CatBoost can naturally handle categorical variables containing an 'Unknown' category.

### Implementation

The '?' values are replaced with the category 'Unknown' in both the training and testing datasets to ensure consistency across the modeling pipeline.

In [60]:
import pandas as pd
import numpy as np
columns = [
    'age','workclass','fnlwgt','education',
    'education_num','marital_status',
    'occupation','relationship','race',
    'sex','capital_gain','capital_loss',
    'hours_per_week','native_country',
    'income'
]
adult_data = pd.read_csv(
    "adult.data",
    header=None,
    names=columns,
    skipinitialspace=True
)
adult_test = pd.read_csv(
    "adult.test",
    header=None,
    names=columns,
    skiprows=1,
    skipinitialspace=True
)

In [61]:
missing_train = pd.DataFrame({
    'Missing Count': (adult_data == '?').sum()
})
missing_train['Missing %'] = round(
    missing_train['Missing Count'] / len(adult_data) * 100,
    2
)
missing_train[missing_train['Missing Count'] > 0]

,Missing Count,Missing %
workclass,1836,5.64
occupation,1843,5.66
native_country,583,1.79


In [62]:
missing_test = pd.DataFrame({
    'Missing Count': (adult_test == '?').sum()})
missing_test['Missing %'] = round(
    missing_test['Missing Count'] / len(adult_test) * 100,2)
missing_test[missing_test['Missing Count'] > 0]

,Missing Count,Missing %
workclass,963,5.91
occupation,966,5.93
native_country,274,1.68


In [63]:
# Replace '?' with 'Unknown'
categorical_missing_cols = [
    'workclass',
    'occupation',
    'native_country'
]
for col in categorical_missing_cols:
    adult_data[col] = adult_data[col].replace('?', 'Unknown')
    adult_test[col] = adult_test[col].replace('?', 'Unknown')

In [64]:
(adult_data == '?').sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,0
relationship,0
race,0
sex,0


In [65]:
(adult_test == '?').sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,0
relationship,0
race,0
sex,0


## Duplicate Record Handling Policy

### Problem Identification

A small number of duplicate records were identified during data validation in both the training and testing datasets.

Although the proportion of duplicates is very low, duplicate observations may introduce bias during model training and evaluation.

### Decision

All duplicate records will be removed from both datasets.

### Justification

- Duplicate records do not provide additional information to the model.
- Repeated observations can disproportionately influence model learning.
- Removing duplicates improves data quality and ensures that each observation contributes only once to the training process.
- The number of duplicate records is very small; therefore, removing them will not significantly reduce the dataset size.

### Implementation

Duplicate records are removed using the `drop_duplicates()` method and the resulting datasets are used for all subsequent preprocessing and modeling steps.

In [66]:
train_duplicates = adult_data.duplicated().sum()
test_duplicates = adult_test.duplicated().sum()
print("Train Duplicates:", train_duplicates)
print("Test Duplicates :", test_duplicates)

Train Duplicates: 24
Test Duplicates : 5


In [67]:
train_dup_pct = round(
    train_duplicates / len(adult_data) * 100,2)
test_dup_pct = round(
    test_duplicates / len(adult_test) * 100,2)
print("Train Duplicate %:", train_dup_pct)
print("Test Duplicate % :", test_dup_pct)

Train Duplicate %: 0.07
Test Duplicate % : 0.03


In [68]:
# Remove duplicate records
adult_data = adult_data.drop_duplicates()
adult_test = adult_test.drop_duplicates()

In [69]:
adult_data.duplicated().sum()
adult_test.duplicated().sum()

np.int64(0)

## Outlier Handling Policy

### Problem Identification

During Exploratory Data Analysis (EDA), several numerical features exhibited statistical outliers when examined using boxplots:

- age
- fnlwgt
- capital_gain
- capital_loss
- hours_per_week

These observations were identified as values lying outside the typical interquartile range (IQR).

### Decision

No outlier removal or capping will be performed.

### Justification

- The identified outliers represent legitimate observations rather than data entry errors.
- Older individuals, high working hours, large capital gains, and large capital losses are realistic values that may carry important predictive information.
- Removing these observations could result in the loss of valuable information and potentially reduce model performance.
- CatBoost is a tree-based algorithm and is generally robust to extreme values and skewed numerical distributions.
- Therefore, retaining these observations is considered the most appropriate strategy for this project.

### Implementation

All numerical features will be retained in their original form without applying outlier removal, winsorization, clipping, or transformation techniques.

In [70]:
# Outliers were identified during EDA using boxplots

numerical_cols = [
    'age',
    'fnlwgt',
    'education_num',
    'capital_gain',
    'capital_loss',
    'hours_per_week'
]
adult_data[numerical_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
age,32537.0,38.585549,13.637984,17.0,28.0,37.0,48.0,90.0
fnlwgt,32537.0,189780.848511,105556.471009,12285.0,117827.0,178356.0,236993.0,1484705.0
education_num,32537.0,10.081815,2.571633,1.0,9.0,10.0,12.0,16.0
capital_gain,32537.0,1078.443741,7387.957424,0.0,0.0,0.0,0.0,99999.0
capital_loss,32537.0,87.368227,403.101833,0.0,0.0,0.0,0.0,4356.0
hours_per_week,32537.0,40.440329,12.346889,1.0,40.0,40.0,45.0,99.0


# Target Variable Encoding Policy

The target variable (`income`) is currently represented as categorical text labels:

- <=50K
- >50K

Additionally, the testing dataset contains target labels with trailing periods:

- <=50K.
- >50K.

Machine learning algorithms require numerical target values for binary classification tasks.

### Decision

The income variable will be encoded as a binary target:

| Original Label | Encoded Value |
|---------------|---------------|
| <=50K | 0 |
| >50K | 1 |

### Justification

- Binary encoding converts the target variable into a machine-learning-friendly format.
- The encoding preserves the original meaning of the classes.
- Numerical target values simplify model training, evaluation, and probability interpretation.
- The same encoding strategy will be applied consistently across both training and testing datasets.

### Implementation

The target variable is converted into binary numerical values where:

- 0 represents income less than or equal to $50K.
- 1 represents income greater than $50K.

In [71]:
adult_data['income'].value_counts()

,count
income,
<=50K,24698
>50K,7839


In [72]:
adult_test['income'].value_counts()

,count
income,
<=50K.,12430
>50K.,3846


In [73]:
adult_data['income'] = adult_data['income'].map({
    '<=50K': 0,
    '>50K': 1
})

In [74]:
adult_test['income'] = adult_test['income'].map({
    '<=50K.': 0,
    '>50K.': 1
})

In [75]:
print('Train_data :\n',adult_data['income'].value_counts())
print('Test_data :\n',adult_test['income'].value_counts())

Train_data :
 income
0    24698
1     7839
Name: count, dtype: int64
Test_data :
 income
0    12430
1     3846
Name: count, dtype: int64


# Categorical Feature Handling Policy

The dataset contains several categorical features, including:

- workclass
- education
- marital_status
- occupation
- relationship
- race
- sex
- native_country

These variables contain textual categories and cannot be directly processed by most machine learning algorithms.

### Decision

No manual encoding technique such as One-Hot Encoding or Label Encoding will be applied.

### Justification

- CatBoost supports categorical variables natively and can process them directly during model training.
- Manual encoding would increase dataset dimensionality and introduce unnecessary complexity.
- Retaining original categorical values allows CatBoost to apply its internal categorical encoding strategy.
- This approach preserves information while simplifying the preprocessing pipeline.

### Implementation

Categorical features will be retained in their original form and explicitly specified as categorical variables during CatBoost model training.

In [76]:
categorical_cols = adult_data.select_dtypes(
    include='object'
).columns.tolist()
categorical_cols

['workclass',
 'education',
 'marital_status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native_country']

In [77]:
categorical_cols = adult_test.select_dtypes(
    include='object'
).columns.tolist()
categorical_cols

['workclass',
 'education',
 'marital_status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native_country']

In [78]:
print("Number of categorical features:",
      len(categorical_cols))

Number of categorical features: 8


In [79]:
# identify the count of unique Values in each coloumn
for col in categorical_cols:
    print(f"{col}")
    print(adult_data[col].nunique())

workclass
9
education
16
marital_status
7
occupation
15
relationship
6
race
5
sex
2
native_country
42


# Feature Scaling Policy

### Problem Identification

The dataset contains several numerical features with different ranges and scales, including:

- age
- fnlwgt
- education_num
- capital_gain
- capital_loss
- hours_per_week

For example, age ranges from 17 to 90, while capital_gain can reach values as high as 99,999.

Many machine learning algorithms require feature scaling to ensure that variables with larger magnitudes do not dominate the learning process.

### Decision

No feature scaling technique such as Standardization (StandardScaler) or Normalization (MinMaxScaler) will be applied.

### Justification

- CatBoost is a tree-based gradient boosting algorithm and does not rely on distance calculations or gradient magnitudes affected by feature scales.
- Tree-based models are inherently insensitive to differences in feature ranges.
- Applying scaling would not provide significant performance benefits and would introduce unnecessary preprocessing complexity.
- Retaining original feature values improves interpretability and simplifies the modeling pipeline.

### Implementation

Numerical features will be retained in their original scale and used directly during CatBoost model training.

In [80]:
# Compare numerical feature scales in train and test datasets
numerical_cols = [
    'age',
    'fnlwgt',
    'education_num',
    'capital_gain',
    'capital_loss',
    'hours_per_week'
]
print("Training Dataset")
display(adult_data[numerical_cols].describe().T)
print("\nTesting Dataset")
display(adult_test[numerical_cols].describe().T)

Training Dataset


,count,mean,std,min,25%,50%,75%,max
age,32537.0,38.585549,13.637984,17.0,28.0,37.0,48.0,90.0
fnlwgt,32537.0,189780.848511,105556.471009,12285.0,117827.0,178356.0,236993.0,1484705.0
education_num,32537.0,10.081815,2.571633,1.0,9.0,10.0,12.0,16.0
capital_gain,32537.0,1078.443741,7387.957424,0.0,0.0,0.0,0.0,99999.0
capital_loss,32537.0,87.368227,403.101833,0.0,0.0,0.0,0.0,4356.0
hours_per_week,32537.0,40.440329,12.346889,1.0,40.0,40.0,45.0,99.0



Testing Dataset


,count,mean,std,min,25%,50%,75%,max
age,16276.0,38.770890,13.849484,17.0,28.0,37.0,48.0,90.0
fnlwgt,16276.0,189442.121959,105708.554555,13492.0,116743.5,177829.5,238384.0,1490400.0
education_num,16276.0,10.072438,2.567570,1.0,9.0,10.0,12.0,16.0
capital_gain,16276.0,1082.237466,7585.077133,0.0,0.0,0.0,0.0,99999.0
capital_loss,16276.0,87.926272,403.164257,0.0,0.0,0.0,0.0,3770.0
hours_per_week,16276.0,40.394507,12.478902,1.0,40.0,40.0,45.0,99.0


# Feature Selection Policy

### Problem Identification

The dataset contains a combination of numerical and categorical features that may contribute to income prediction.

Certain features warrant consideration before model training:

- education and education_num contain related information.
- fnlwgt represents a census sampling weight rather than a direct demographic characteristic.

Feature selection decisions must balance model simplicity against the risk of losing potentially useful predictive information.

### Decision

All available predictor features will be retained for model training.

### Justification

- CatBoost is capable of handling both numerical and categorical features effectively.
- Premature feature removal may eliminate information that contributes to predictive performance.
- Although education and education_num are related, they provide information in different formats (categorical and numerical).
- The predictive usefulness of fnlwgt will be evaluated through feature importance analysis after model training.
- Retaining all features allows the model to determine which variables contribute most to income prediction.

### Implementation

All predictor variables will be included during CatBoost model training. Feature importance analysis will be used later to evaluate the relative contribution of each feature.

In [81]:
# Features selected for model training
feature_columns = [
    'age',
    'workclass',
    'fnlwgt',
    'education',
    'education_num',
    'marital_status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'capital_gain',
    'capital_loss',
    'hours_per_week',
    'native_country'
]
print("Number of Predictor Features:", len(feature_columns))
for feature in feature_columns:
    print( feature)
print("Target Variable: income")

Number of Predictor Features: 14
age
workclass
fnlwgt
education
education_num
marital_status
occupation
relationship
race
sex
capital_gain
capital_loss
hours_per_week
native_country
Target Variable: income


In [82]:
# Save cleaned datasets
adult_data.to_csv(
    'adult_train_clean.csv',
    index=False
)
adult_test.to_csv(
    'adult_test_clean.csv',
    index=False
)